### Transform Refund Data
1. Extract specific portion of the string from refund_reason using split fundtion
2. Extract specific portion of the string from refund_reason using regexp_extract function
3. Extract date and time from the refund_timestamp
4. Write transformed data to the silver schema in hive metastore

In [0]:
df_refunds = spark.table('gizmobox.bronze.py_refunds')

display(df_refunds)

## 1. Extract specific portion of the string from refund_reason using split fundtion

In [0]:

from pyspark.sql import functions as F

df_split_refund = (
                    df_refunds
                    .select(
                        "refund_id",
                        "payment_id",
                        "refund_amount",
                        "refund_timestamp",
                        F.split('refund_reason',':')[0].alias('refund_reason'),
                        F.split('refund_reason',':')[1].alias('refund_source')
                        )
)

display(df_split_refund)

In [0]:
df_transformed_refunds = (
                df_refunds
                .select(
                    'refund_id',
                    'payment_id',
                    'refund_amount',
                    F.date_format('refund_timestamp','yyyy-MM-dd').cast("date").alias('payment_date'),
                    F.date_format('refund_timestamp','HH:mm:ss').alias('payment_time'),
                    F.regexp_extract("refund_reason",'^([^:]+):',1).alias('refund_reason'),
                    F.regexp_extract("refund_reason",'^[^:]+:(.*)$',1).alias('refund_source')
                )
)

display(df_transformed_refunds)

## 3. Write transformed data to silver schema

In [0]:
df_transformed_refunds.writeTo('gizmobox.silver.py_refunds').createOrReplace()

In [0]:
%sql
select * from gizmobox.silver.py_refunds